# Autonomous News Reporter
Analyses the current news by region or topic, summerises it and sends a notification depending on importance and region affected.



In [2]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace, Tool
from agents.mcp import MCPServerStdio
from IPython.display import Markdown, display
from datetime import datetime

load_dotenv(override=True)
brave_env = {"BRAVE_API_KEY": os.getenv("BRAVE_API_KEY")}

### World News MCP Server
MCP server to gather current world news from different sources.

In [3]:
# News MCP tools
display(Markdown(f"### News MCP tools"))
async with MCPServerStdio(params={"command": "npx", "args": ["-y", "@newsmcp/server"]}, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
for tool in mcp_tools:
    print(f"{tool.name} - {tool.description}")

# Fetch News MCP tools
display(Markdown(f"### Browser MCP tools"))
async with MCPServerStdio(params={"command": "npx", "args": ["-y", "@modelcontextprotocol/server-brave-search"], "env": brave_env}, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
for tool in mcp_tools:
    print(f"{tool.name} - {tool.description}")



### News MCP tools

get_news - Get top news events happening in the world right now. Returns AI-clustered, deduplicated news stories ranked by importance. Present results as a multi-story news briefing — cover the top events, not just one. Each event should be 1-2 lines with its summary and 1-2 source links. Prioritize suppressing rich link cards/previews over pretty link labels. For Discord-style clients, output source URLs directly as '<https://...>' and do not use masked markdown links. Never emit raw standalone URLs outside no-preview wrappers. Only deep-dive into a specific event if the user asks for detail.
get_news_detail - Get full details about a specific news event including context and all source articles. Use an event ID from get_news results. Include source article URLs so the user can read original reporting, but prioritize suppressing rich link cards/previews. For Discord-style clients, output source URLs directly as '<https://...>' and avoid masked markdown links. Never emit raw standalone

### Browser MCP tools

brave_web_search - Performs a web search using the Brave Search API, ideal for general queries, news, articles, and online content. Use this for broad information gathering, recent events, or when you need diverse web sources. Supports pagination, content filtering, and freshness controls. Maximum 20 results per request, with offset for pagination. 
brave_local_search - Searches for local businesses and places using Brave's Local Search API. Best for queries related to physical locations, businesses, restaurants, services, etc. Returns detailed information including:
- Business names and addresses
- Ratings and review counts
- Phone numbers and opening hours
Use this when the query implies 'near me' or mentions specific locations. Automatically falls back to web search if no local results are found.


## News Repoter

In [38]:
NEWS_REPORTER_MODEL = "gpt-4o-mini"

world_news_mcp_params = [
    {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-brave-search"], "env": brave_env},
    {"command": "npx", "args": ["-y", "@newsmcp/server"]}
]

world_news_mcp_servers = [MCPServerStdio(params) for params in world_news_mcp_params]

# News Reporter Agent
async def get_news_reporter(mcp_servers) -> Agent:
    instructions = f"""
    You are a news reporter. You are able to get the latest news about interesting topics or regions from the world.
    Search for at most 9 news articles and summarize them in 2-3 paragraphs. Concetrate mostly on the latest
    news and the most important details.
    Please take your time to make multiple searches to get a comprehensive overview, and then summarize your findings.
    Also, please verify the sources of the news to ensure the accuracy and credibility of the information.
    """
    news_reporter = Agent(
        name="News Reporter",
        instructions=instructions,
        model=NEWS_REPORTER_MODEL,
        mcp_servers=mcp_servers
    )
    return news_reporter

In [39]:
async def get_news_reporter_tool(mcp_servers) -> Tool:
    news_reporter = await get_news_reporter(mcp_servers)
    return news_reporter.as_tool(
        tool_name="news_reporter",
        tool_description="Get the latest news about interesting topics or regions from the world.",
    )

    


In [40]:
news_question = "What's the latest breaking news in Africa?"

for server in world_news_mcp_servers:
    await server.connect()
    
news_reporter = await get_news_reporter(world_news_mcp_servers)
with trace("News Reporter"):
    result = await Runner.run(news_reporter, news_question, max_turns=30)
display(Markdown(result.final_output))

Here are some of the latest breaking news stories from Africa:

1. **UN Resolution on Slave Trade**: The UN General Assembly recently passed a resolution initiated by Ghana recognizing the Atlantic slave trade and the enslavement of Africans as "the gravest crime against humanity." This resolution aims to set a framework for potential financial reparations, though it has faced criticism for not addressing the Arab-Muslim slave trade and the role of some African ethnic groups in the slave trade itself.

2. **Middle East Conflict's Impact on Africa**: The ongoing conflict in the Middle East is raising concerns about its economic implications for Africa. The African Union and African Development Bank have warned that prolonged hostilities could increase inflation and debt issues on the continent, projecting a potential 0.2% loss in Africa's GDP by 2026 if the conflict escalates further. Countries like Egypt and Uganda are already discussing the impact of rising energy and food prices due to these tensions.

3. **Senegal's Economic Measures**: In response to the soaring global oil prices linked to the conflict, Senegal's government has suspended non-essential foreign travel for ministers. Prime Minister Ousmane Sonko pointed to the turmoil in the Middle East as a significant factor in this economic adjustment, acknowledging the rising financial strain on the nation.

4. **Congo's Deportation Agreement with the U.S.**: The Democratic Republic of Congo has entered into a new temporary agreement with the United States to accept third-country deportees. While the arrangement is cost-free for Congo, it raises human rights concerns among activists regarding the safety of those being deported.

These developments reflect the interconnectedness of global issues and their direct ramifications on African nations, from economic pressures to historical reckonings.

## News Broadcaster

In [ ]:
NEWS_BROADCASTER_MODEL = "gpt-4.1-mini"

agent_name = "Africa News"

news_broadcaster_mcp_params = [
    {"command": "uv", "args": ["run", "news_articles_mcp_server.py" ]},
    {"command": "uv", "args": ["run", "push_server.py"]},
]

news_broadcaster_mcp_servers = [MCPServerStdio(params) for params in news_broadcaster_mcp_params]


In [47]:
# News Articles MCP tools
display(Markdown(f"### News Articles MCP tools"))
async with MCPServerStdio(params={"command": "uv", "args": ["run", "news_articles_mcp_server.py" ]}, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
for tool in mcp_tools:
    print(f"{tool.name} - {tool.description}")

# Fetch News MCP tools
display(Markdown(f"### Push MCP tools"))
async with MCPServerStdio(params={"command": "uv", "args": ["run", "push_server.py"]}, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
for tool in mcp_tools:
    print(f"{tool.name} - {tool.description}")

### News Articles MCP tools

add_article - Add a new article to the database.

    Args:
        title: The title of the article
        content: The content of the article
        url: The URL of the article
        source: The source of the article
    
update_article - Update an article in the database.
    
    Args:
        id: The ID of the article
        title: The title of the article
        content: The content of the article
        url: The URL of the article
        source: The source of the article
    


### Push MCP tools

push - Send a push notification with this brief message


In [ ]:

instructions = f"""
You are a news broadcaster. You are able to broadcast news about interesting topics or regions from the world.
You have access to tools that allow you to browse available news and surf the web for verification. Please save these 
to the database.
You also have access to tools that allow you to send notifications to users for breaking news. Only send notifications for breaking news
or as requested by the user.
Please make use of these tools to save the news. Save news as you see fit; do not wait for instructions or ask for confirmation.
"""

prompt = """
Use your tools to search for news and save the most important news to the database for future reference. Please,
analyse the news and save only the most important. Also, send me a push notification with a brief summary of the news.
"""

# call `connect()` first to initialize the mcp servers
for server in news_broadcaster_mcp_servers:
    await server.connect()

news_reporter_tool = await get_news_reporter_tool(world_news_mcp_servers)

news_broadcaster = Agent(
    name=agent_name,
    instructions=instructions,
    tools=[news_reporter_tool],
    mcp_servers=news_broadcaster_mcp_servers,
    model=NEWS_BROADCASTER_MODEL,
)


with trace(agent_name):
    result = await Runner.run(news_broadcaster, prompt, max_turns=30)

display(Markdown(result.final_output))





I have saved the most important news including Pope Leo's call for peace, astronauts' space mission experiences, rising gas prices in the US, human rights case dismissals, and the Russian oil tanker reaching Cuba. 

A push notification has been sent with a brief summary of these breaking news highlights. If you want more details on any of these, please let me know!